In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import numpy as np
import pandas as pd

from transformers import pipeline
from sklearn.metrics import classification_report
import time


In [2]:
#Electronics Dataset:

fileElectr='amazon_reviews_us_Electronics_v1_00.tsv'
df=pd.read_csv(fileElectr, sep="\t", header=0, on_bad_lines='skip')
df=df.dropna(subset=['review_headline', 'review_body', 'star_rating'])




In [3]:
from transformers import pipeline


In [4]:
#Model 12 (siebert) Helper Functions

In [5]:
#https://huggingface.co/siebert/sentiment-roberta-large-english
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("siebert/sentiment-roberta-large-english")

model = AutoModelForSequenceClassification.from_pretrained("siebert/sentiment-roberta-large-english")

In [6]:
def sentiment_classify(df_sample, column_name):
    sentiment_analysis = pipeline("sentiment-analysis",model="siebert/sentiment-roberta-large-english")

    for i in range (0, len(df_sample[column_name])):

        text=df_sample[column_name][i]                 
        
        if len(text) > 514:
            text = text[:514]
            
        prediction={}
        prediction = sentiment_analysis(text)

        if prediction[0]['label']=='POSITIVE':
            df_sample.loc[i, ("sentiment_analysis")]=5

        elif prediction[0]['label']== 'NEGATIVE':
            df_sample.loc[i, ("sentiment_analysis")]=1
        else: 
            print("Error.")
            
      #  if i%1000==0:
      #     print ("\n sentiment_classify:   We are at i=", str(i))    
            
            
    return df_sample
            

In [7]:
#Model 11 / Ref cell 43 at https://github.com/sophiej-s/MSThesis-I/blob/main/wk15.ipynb
#Model 11  Helper Functions

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
stop_words = set(stopwords.words("english"))
from nltk.stem import PorterStemmer
from nltk.stem.wordnet import WordNetLemmatizer
lemma = WordNetLemmatizer()
ps = PorterStemmer()
import re


from sklearn.model_selection import train_test_split

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix





[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/sophie/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [9]:
#from transformers import pipeline

#classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", return_all_scores=True)

def emotion_roberta(df_sample, column_name):
    classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", return_all_scores=True)


    for i in range (0, len(df_sample[column_name])):

        #processed text
        text=df_sample[column_name][i]  

        if len(text) > 512:
            text = text[:512]

        prediction = classifier(text )

        for j in range(0,7):  #loop over the  emotions:
            df_sample.loc[i, ("roberta_"+column_name+prediction[0][j]['label'])]=prediction[0][j]['score'] #BP=body processed


       # if i%1000==0:
       #    print ("\n emotion_roberta:   We are at i=", str(i))    
            
            
    return df_sample

In [10]:
def text_process2(reviews, column_name):  #input is the dataframe
    for i  in range(0, reviews[column_name].count()):
       review_body=reviews.loc[i, (column_name)]  #tokens= word_tokenize(df_sample.loc[1, ('review_body')])
       review_body=re.sub('<br\s?\/>|<br>', " ", review_body)  #remove the br
       tokens= word_tokenize(review_body)
       tokens = [w.lower()  for w in tokens ]
       #tokens = [w for w in tokens if not w in stop_words]
       tokens = [w for w in tokens if w.isalpha()] #remove non alphabetic items like like 5 or ;
       tokens = [lemma.lemmatize(w) for w in tokens]
       # tokens = [ps.stem(w) for w in tokens]
       column_name_out=column_name+"_processed"
       reviews.loc[i, (column_name_out)]=' '.join(tokens)
       
      # if i%10000==0:
      #     print ("\n text_process:   We are at i=", str(i))
       
    return reviews

In [11]:

Roberta_Body=['roberta_review_bodyanger',
'roberta_review_bodydisgust',
'roberta_review_bodyfear',
'roberta_review_bodyjoy',
'roberta_review_bodyneutral',
'roberta_review_bodysadness',
'roberta_review_bodysurprise']



Roberta_Head=[
'roberta_review_headlineanger', 
'roberta_review_headlinedisgust',
'roberta_review_headlinefear', 
'roberta_review_headlinejoy',
'roberta_review_headlineneutral', 
'roberta_review_headlinesadness',
'roberta_review_headlinesurprise']


Roberta_Head_Processed=[
'roberta_review_headline_processedanger', 
'roberta_review_headline_processeddisgust',
'roberta_review_headline_processedfear', 
'roberta_review_headline_processedjoy',
'roberta_review_headline_processedneutral', 
'roberta_review_headline_processedsadness',
'roberta_review_headline_processedsurprise'] 

In [12]:
def run_SVC(input_df,input_y):
    target_names = ['0 = rating of 1',  '1 = rating of 5'] # 0 = negative, 4 = positive

    X_train, X_test, y_train, y_test = train_test_split(input_df, input_y['star_rating'], test_size=0.33, random_state=1726)
    
    #{'C': 1000, 'gamma': 0.001, 'kernel': 'rbf'}

    clf = make_pipeline(StandardScaler(), SVC( C=1000, gamma= 0.001, kernel= 'rbf'))
    clf.fit(X_train, y_train)
    y_pred=clf.predict(X_test)
    
    return y_pred,y_test

    #clf.score(X_test, y_test)
    #y_test.value_counts()
   # print(clf.score(X_test, y_test))

#    print(classification_report(y_test, y_pred, target_names=target_names, digits=6))

#    print(confusion_matrix(y_test, y_pred))

In [13]:
#adding embeddings 
def df2emd(word2vec_model, N_rewiews, column_name):
    word2vec_model_embeddings = WordVecVectorizer(word2vec_model)

    word2vec_model_embeddings_ave_one_review_list=[]
    embed_only=pd.DataFrame()
    
    for i in range(0,len(N_rewiews[column_name]) ) : 
    #for i in range(0,len(N_rewiews['review_body_process']) ) : 
        #list_words=[N_rewiews['review_body_process'][i]]
        list_words=[N_rewiews[column_name][i]]

        list_words=check_against_word2vec_model(list_words, word2vec_model)
        word2vec_embeddings_one_review=word2vec_model_embeddings.transform(list_words)
        word2vec_model_embeddings_ave_one_review_list.append(word2vec_embeddings_one_review)

    embed_only=pd.DataFrame(np.concatenate(word2vec_model_embeddings_ave_one_review_list))
    
    return embed_only

In [14]:
class WordVecVectorizer(object):
    def __init__(self, word2vec_model):
        self.word2vec_model = word2vec_model
        self.dim = 300
    def transform(self, X):
        return np.array([
            np.mean([self.word2vec_model[w] for w in texts.split() if w in self.word2vec_model]
                    or [np.zeros(self.dim)], axis=0)
            for texts in X
        ])

def check_against_word2vec_model(list_topics, word2vec_model):
    for i  in range(0, len(list_topics) ):
       tokens= word_tokenize(list_topics[i])
       tokens = [w for w in tokens if w in word2vec_model.key_to_index ]
       list_topics[i]=' '.join(tokens)
       return list_topics

In [15]:
import gensim

file_embeddings_fast='crawl-300d-2M.vec'

word2vec_model_fast = gensim.models.KeyedVectors.load_word2vec_format(file_embeddings_fast) 
print(word2vec_model_fast.vector_size)

300


In [16]:
#Running Designs 11 and 12 

In [17]:
#Sampling the dataset


import random
random.seed(a=12, version=2)


for i in range(0, 3):
    
    random_state=random.randint(0, 10000)
    print("random_state: ",random_state) #print random_state for reproducibility

    
    n_samples=10000 #LARGER SIZE than in the previous cell
    N_rewiews=df.loc[df['star_rating'] == 1].sample(n_samples, replace=False, random_state=random_state)
    N_rewiew2=df.loc[df['star_rating'] == 5].sample(n_samples, replace=False, random_state=random_state)

    samplesize=n_samples*2
    N_rewiews=N_rewiews.append(N_rewiew2)

    N_rewiews=N_rewiews.reset_index()
    N_rewiews['star_rating'].value_counts()


    #===================================
    #Running Model 11

    t_START = time.time()
    N_rewiews=text_process2(N_rewiews,'review_headline')

    emotion_roberta(N_rewiews, 'review_body')

    emotion_roberta(N_rewiews, 'review_headline')

    emotion_roberta(N_rewiews, 'review_headline_processed')
    df_sample_all_test=N_rewiews[ Roberta_Body+Roberta_Head+Roberta_Head_Processed ] 

    embed_only_fast_B=df2emd(word2vec_model_fast, N_rewiews, "review_body")
    embed_only_fast_HP=df2emd(word2vec_model_fast, N_rewiews, "review_headline")

    embed_combined=embed_only_fast_B.join(embed_only_fast_HP, lsuffix='_caller', rsuffix='_other')

    #combine the embeddings with the emotions
    embed_emptions_combined=embed_combined.join(df_sample_all_test, lsuffix='_caller', rsuffix='_other')


    #{'C': 1000, 'gamma': 0.001, 'kernel': 'rbf'}
    y_pred, y_test=run_SVC(embed_emptions_combined,N_rewiews)

    elapsed = time.time() - t_START
    print("Design 11 elapsed Time is:  ", elapsed)


    #Evaluation step:
    target_names = ['0 = rating of 1',  '1 = rating of 5'] 
    print(classification_report(y_test, y_pred, target_names=target_names, digits=6))

    #print(clf.score(X_test, y_test))
    #print(confusion_matrix(y_test, y_pred))

    #===================================
    #Running Model 12 (siebert)
    t_START = time.time()
    sentiment_classify(N_rewiews, 'review_body')
    elapsed = time.time() - t_START
    print("Design 12 elapsed Time is:  ", elapsed)


    #Evaluation step:
    y_pred= N_rewiews['sentiment_analysis']
    y_test=N_rewiews['star_rating']

    target_names = ['0 = rating of 1',  '1 = rating of 5'] 
    print(classification_report(y_test, y_pred, target_names=target_names, digits=6))


    
    
    #===================================







random_state:  7775
Design 11 elapsed Time is:   4358.5222589969635
                 precision    recall  f1-score   support

0 = rating of 1   0.946702  0.950621  0.948657      3382
1 = rating of 5   0.947878  0.943754  0.945811      3218

       accuracy                       0.947273      6600
      macro avg   0.947290  0.947187  0.947234      6600
   weighted avg   0.947275  0.947273  0.947270      6600

Design 12 elapsed Time is:   21000.452919960022
                 precision    recall  f1-score   support

0 = rating of 1   0.970615  0.967800  0.969205     10000
1 = rating of 5   0.967893  0.970700  0.969295     10000

       accuracy                       0.969250     20000
      macro avg   0.969254  0.969250  0.969250     20000
   weighted avg   0.969254  0.969250  0.969250     20000

random_state:  4407
Design 11 elapsed Time is:   4387.690896272659
                 precision    recall  f1-score   support

0 = rating of 1   0.949477  0.939089  0.944254      3382
1 = rating o